# Surper_GCG Colab Runner

This notebook runs the current `poc/run_pipeline.py` entrypoint directly.

Recommended order:
1. Run `baseline_diagnosis` first to open the real `12B` mainline
2. Treat `eval_calibration` / `Exp11` as an optional legacy review-pack side quest
3. Read the exported artifacts before advancing to later mechanism stages
4. Outputs are written to `poc/results/pipeline_runs/`


## 1. Mount Drive

Use Google Drive to persist the repo checkout and the pipeline outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repo And Install

Clone the repository into `/content`, then install the package from the `poc` directory.

This cell is idempotent: if the repo already exists in the runtime, it will skip cloning.

In [ ]:
%cd /content
import os
if not os.path.exists('/content/Surper_GCG'):
    !git clone https://github.com/awaa-col/Surper_GCG.git
else:
    print('Repo already exists at /content/Surper_GCG; skipping clone.')
%cd /content/Surper_GCG/poc
!pip install -e .

## 3. Configure Runtime

Set Hugging Face credentials, run name, model, and shared pipeline knobs.

In [ ]:
import os

HF_TOKEN = ''
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

RUN_NAME = 'gemma3_12b_main'
RESUME = '--resume'  # set to '' for a fresh run
MODEL = 'google/gemma-3-12b-it'
N_TRAIN = 64
N_EVAL = 2
MAX_NEW_TOKENS = 96


## 4. Inspect Available Pipeline Presets And Stages

List the current presets first, then inspect which stages are runnable versus blocked.

In [ ]:
%cd /content/Surper_GCG/poc
!python run_pipeline.py --list-presets

In [ ]:
!python run_pipeline.py --list-stages

## 5. Dry-Run The Real Frontline Tech

Start with `baseline_diagnosis`. This is the first real `12B` frontline tech point.

Use dry-run first to confirm the selected preset, output directory, and generated commands.

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 6. Run The Frontline Pipeline

Default mainline: run `baseline_diagnosis` first.

Do not treat `eval_calibration` as the real first battle. It is only a support tech for review alignment.

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$RUN_NAME" \
  $RESUME \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 6B. Optional Side Quest: Legacy Review-Pack Calibration

Run this only if you intentionally want to build a manual review pack from legacy saved results.

This is not the same thing as starting `12B` from scratch.

In [ ]:
# Optional: only run if your Colab environment already includes the legacy result files needed by Exp11
!python run_pipeline.py \
  --preset eval_calibration \
  --run-name "${RUN_NAME}_eval_calibration" \
  $RESUME \
  --model "$MODEL"

In [ ]:
print('Current unlock target: confirm the raw 12B refusal object and baseline seams. Only after that should you move to gate discovery refactors.')

## 7. Inspect Outputs And Save A Zip To Drive

Inspect the run directory, then compress exactly this run so it can be downloaded later.

In [ ]:
!ls -lah /content/Surper_GCG/poc/results/pipeline_runs
!ls -lah /content/Surper_GCG/poc/results/pipeline_runs/$RUN_NAME

In [ ]:
!zip -r /content/drive/MyDrive/${RUN_NAME}_pipeline_results.zip /content/Surper_GCG/poc/results/pipeline_runs/$RUN_NAME